# 📡 TelecomX – Parte 2: Predicción de Cancelación (Churn)

Este cuaderno continúa el trabajo de la **Parte 1 (ETL)** y construye modelos de Machine Learning para predecir qué clientes tienen mayor probabilidad de cancelar sus servicios.

**Objetivos:**
- Preparar los datos para modelado (codificación, normalización)
- Analizar correlaciones y seleccionar variables
- Entrenar y evaluar modelos de clasificación
- Interpretar resultados con insights estratégicos

## 0. Importación de Librerías

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Librerías cargadas correctamente")

## 1. Extracción y Transformación de Datos (ETL)

> **Nota:** Si ya ejecutaste la Parte 1 y tienes `datos_tratados.csv`, puedes saltar a la sección 2. Este bloque reproduce el ETL completo desde el JSON original para garantizar reproducibilidad.

In [ ]:
# Lectura del JSON original
with open('TelecomX_Data.json') as f:
    data = json.load(f)

# Aplanado de la estructura anidada
rows = []
for record in data:
    row = {
        'customerID': record.get('customerID'),
        'Churn': record.get('Churn'),
    }
    for section in ['customer', 'phone', 'internet', 'account']:
        section_data = record.get(section, {})
        for k, v in section_data.items():
            if isinstance(v, dict):
                for k2, v2 in v.items():
                    row[k2] = v2
            else:
                row[k] = v
    rows.append(row)

df_raw = pd.DataFrame(rows)
print(f"Registros cargados: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]}")
df_raw.head(3)

In [ ]:
# Limpieza de datos
# 1. Convertir 'Total' a numérico
df_raw['Total'] = pd.to_numeric(df_raw['Total'], errors='coerce')

# 2. Filtrar filas con Churn ambiguo (vacío)
df_raw = df_raw[df_raw['Churn'].isin(['Yes', 'No'])].copy()

# 3. Convertir Churn a binario
df_raw['Churn'] = (df_raw['Churn'] == 'Yes').astype(int)

# 4. Eliminar filas con nulos
df_clean = df_raw.dropna().copy()

print(f"Registros después de limpieza: {df_clean.shape[0]:,}")
print(f"Nulos restantes: {df_clean.isnull().sum().sum()}")
df_clean.info()

In [ ]:
# Guardar datos tratados para reutilización
df_clean.to_csv('datos_tratados.csv', index=False)
print("✅ Datos tratados guardados en 'datos_tratados.csv'")

## 2. Análisis Exploratorio (EDA)

### 2.1 Distribución de la Variable Objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conteo absoluto
churn_counts = df_clean['Churn'].value_counts()
churn_labels = ['Activo (0)', 'Canceló (1)']
axes[0].bar(churn_labels, churn_counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de Cancelación', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad de Clientes')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontweight='bold')

# Proporción
axes[1].pie(churn_counts.values, labels=churn_labels, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción de Churn', fontsize=13, fontweight='bold')

plt.suptitle('Variable Objetivo: Churn de Clientes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

churn_rate = df_clean['Churn'].mean() * 100
print(f"\n📊 Tasa de cancelación: {churn_rate:.1f}%")
print(f"⚠️  Desbalance de clases detectado: ratio aproximado 1:{churn_counts[0]/churn_counts[1]:.1f}")

### 2.2 Variables Numéricas vs Churn

In [ ]:
num_cols = ['tenure', 'Monthly', 'Total', 'SeniorCitizen']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

colors = ['#2ecc71', '#e74c3c']
labels = ['Activo', 'Canceló']

for i, col in enumerate(num_cols):
    for churn_val, color, label in zip([0, 1], colors, labels):
        subset = df_clean[df_clean['Churn'] == churn_val][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(f'Distribución de {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Densidad')
    axes[i].legend()

plt.suptitle('Variables Numéricas por Grupo de Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('num_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.3 Tiempo de Contrato y Gasto Total vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot: Tenure vs Churn
df_plot = df_clean.copy()
df_plot['Churn_label'] = df_plot['Churn'].map({0: 'Activo', 1: 'Canceló'})

for ax, col, title in zip(axes, ['tenure', 'Total'],
                           ['Meses de Permanencia (tenure)', 'Gasto Total (Total)']):
    data_0 = df_plot[df_plot['Churn_label'] == 'Activo'][col]
    data_1 = df_plot[df_plot['Churn_label'] == 'Canceló'][col]
    bp = ax.boxplot([data_0, data_1], labels=['Activo', 'Canceló'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#2ecc71', alpha=0.7),
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][1].set_facecolor('#e74c3c')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(col)

    # Anotaciones estadísticas
    t_stat, p_val = stats.ttest_ind(data_0, data_1)
    ax.text(0.02, 0.97, f'p-value: {p_val:.2e}', transform=ax.transAxes,
            fontsize=9, va='top', color='navy')

plt.suptitle('Análisis de Variables Clave vs Cancelación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_churn.png', dpi=120, bbox_inches='tight')
plt.show()
print("💡 Clientes que cancelaron tienden a tener menor permanencia y menor gasto total.")

### 2.4 Variables Categóricas vs Churn

In [ ]:
cat_cols_plot = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols_plot):
    ct = pd.crosstab(df_clean[col], df_clean['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'],
            edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'Churn por {col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('% de clientes')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=20)
    axes[i].legend(['Activo', 'Canceló'])

plt.suptitle('Tasa de Cancelación por Variable Categórica', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cat_churn.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Preparación de Datos para Modelado

### 3.1 Eliminación de Columnas No Informativas

In [ ]:
# Eliminar ID del cliente (no aporta valor predictivo)
df_model = df_clean.drop(columns=['customerID']).copy()
print(f"Columnas eliminadas: ['customerID']")
print(f"Dimensiones del dataset de modelado: {df_model.shape}")
df_model.head(3)

### 3.2 Codificación de Variables Categóricas

Se utiliza `pd.get_dummies` (equivalente a One-Hot Encoding) para convertir variables categóricas a numéricas. Se elimina una categoría por variable para evitar multicolinealidad.

In [ ]:
# Identificar columnas categóricas
cat_cols = df_model.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols_model = df_model.select_dtypes(include=[np.number]).columns.tolist()

print(f"Variables categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Variables numéricas ({len(num_cols_model)}): {num_cols_model}")

In [ ]:
# One-Hot Encoding
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Convertir booleanos a enteros
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"Dimensiones después de codificación: {df_encoded.shape}")
print(f"Nuevas columnas generadas: {df_encoded.shape[1] - df_model.shape[1]}")
df_encoded.head(3)

### 3.3 Matriz de Correlación

In [ ]:
plt.figure(figsize=(16, 12))
corr_matrix = df_encoded.corr()

# Mask triangulo superior
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={"shrink": 0.6})
plt.title('Matriz de Correlación - Variables del Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Top correlaciones con Churn
churn_corr = corr_matrix['Churn'].drop('Churn').abs().sort_values(ascending=False)
print("\n🔝 Top 10 variables más correlacionadas con Churn:")
print(churn_corr.head(10).to_string())

In [ ]:
# Visualización de top correlaciones con Churn
top_corr = corr_matrix['Churn'].drop('Churn').sort_values()
fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ['#e74c3c' if x > 0 else '#2ecc71' for x in top_corr.values]
top_corr.plot(kind='barh', ax=ax, color=colors_bar)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de Variables con Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente de Correlación')
plt.tight_layout()
plt.savefig('churn_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.4 Separación de Variables y División Train/Test

In [ ]:
# Variables explicativas y objetivo
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nDistribución objetivo:\n{y.value_counts().to_string()}")

In [ ]:
# División 70/30 con estratificación para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=5
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]:,} registros ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Conjunto de prueba:        {X_test.shape[0]:,} registros ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nChurn en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Churn en prueba:         {y_test.mean()*100:.1f}%")

### 3.5 Normalización para Modelos Basados en Distancia

La **Regresión Logística** y **KNN** son sensibles a la escala de las variables. Se aplica `StandardScaler` para estandarizar los datos (media=0, desviación estándar=1). Los modelos basados en árboles (**Decision Tree**, **Random Forest**) no requieren esta transformación.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("✅ Datos normalizados con StandardScaler")
print(f"   Media en train (primeras 3 features): {X_train_sc[:, :3].mean(axis=0).round(4)}")
print(f"   Std  en train (primeras 3 features): {X_train_sc[:, :3].std(axis=0).round(4)}")

## 4. Entrenamiento de Modelos

### 4.1 Modelo Base (Baseline – DummyClassifier)

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=5)
dummy.fit(X_train, y_train)
dummy_acc = dummy.score(X_test, y_test)
print(f"📊 Exactitud del modelo base (Dummy): {dummy_acc:.4f}")
print("   Este es el mínimo que debe superar cualquier modelo real.")

### 4.2 Regresión Logística (requiere normalización)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=5, C=1.0)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=['Activo', 'Canceló']))

### 4.3 Árbol de Decisión (no requiere normalización)

In [ ]:
# Sin limitar profundidad (puede tener overfitting)
tree_full = DecisionTreeClassifier(random_state=5)
tree_full.fit(X_train, y_train)
print(f"Árbol sin límite - Train: {tree_full.score(X_train, y_train):.4f} | Test: {tree_full.score(X_test, y_test):.4f}")
print("⚠️  Alta diferencia train/test indica overfitting")

# Con límite de profundidad
tree = DecisionTreeClassifier(max_depth=5, random_state=5)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
print(f"\nÁrbol max_depth=5 - Train: {tree.score(X_train, y_train):.4f} | Test: {tree.score(X_test, y_test):.4f}")
print("\n=== Árbol de Decisión (max_depth=5) ===")
print(classification_report(y_test, y_pred_tree, target_names=['Activo', 'Canceló']))

In [ ]:
# Visualización del árbol
plt.figure(figsize=(20, 8))
plot_tree(tree, max_depth=3, filled=True, class_names=['Activo', 'Canceló'],
          feature_names=X.columns.tolist(), fontsize=7, impurity=False, proportion=True)
plt.title('Árbol de Decisión (profundidad máx. 3 niveles visualizados)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.4 Random Forest (no requiere normalización)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=5, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"Random Forest - Train: {rf.score(X_train, y_train):.4f} | Test: {rf.score(X_test, y_test):.4f}")
print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['Activo', 'Canceló']))

### 4.5 KNN (requiere normalización)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)

print("=== KNN (k=11) ===")
print(classification_report(y_test, y_pred_knn, target_names=['Activo', 'Canceló']))

## 5. Evaluación y Comparación de Modelos

In [ ]:
def get_metrics(y_true, y_pred, model_name):
    return {
        'Modelo': model_name,
        'Exactitud': accuracy_score(y_true, y_pred),
        'Precisión': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred)
    }

results = [
    get_metrics(y_test, dummy.predict(X_test), 'Dummy (Baseline)'),
    get_metrics(y_test, y_pred_lr,   'Regresión Logística'),
    get_metrics(y_test, y_pred_tree, 'Árbol de Decisión'),
    get_metrics(y_test, y_pred_rf,   'Random Forest'),
    get_metrics(y_test, y_pred_knn,  'KNN (k=11)'),
]

df_results = pd.DataFrame(results).set_index('Modelo')
df_results = df_results.sort_values('F1-Score', ascending=False)
print(df_results.round(4).to_string())

In [ ]:
# Visualización comparativa de métricas
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_results))
width = 0.2
metrics_cols = ['Exactitud', 'Precisión', 'Recall', 'F1-Score']
colors_m = ['#3498db', '#9b59b6', '#e67e22', '#e74c3c']

for i, (metric, color) in enumerate(zip(metrics_cols, colors_m)):
    bars = ax.bar(x + i*width, df_results[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(df_results.index, rotation=15, ha='right')
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('Score')
ax.set_title('Comparación de Modelos por Métricas de Evaluación', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(y=df_results.loc['Dummy (Baseline)', 'Exactitud'], color='gray',
           linestyle='--', alpha=0.5, label='Baseline')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

### 5.1 Matrices de Confusión

In [ ]:
models_preds = [
    ('Regresión Logística', y_pred_lr),
    ('Árbol de Decisión',   y_pred_tree),
    ('Random Forest',       y_pred_rf),
    ('KNN (k=11)',          y_pred_knn),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, y_pred) in zip(axes, models_preds):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Activo', 'Canceló'],
                yticklabels=['Activo', 'Canceló'],
                cbar=False, linewidths=0.5)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.suptitle('Matrices de Confusión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

### 5.2 Análisis de Overfitting / Underfitting

In [ ]:
print("📊 Análisis Train vs Test (detección de Overfitting)\n")
train_test_compare = {
    'Regresión Logística': (lr.score(X_train_sc, y_train), lr.score(X_test_sc, y_test)),
    'Árbol (sin límite)':  (tree_full.score(X_train, y_train), tree_full.score(X_test, y_test)),
    'Árbol (max_depth=5)': (tree.score(X_train, y_train), tree.score(X_test, y_test)),
    'Random Forest':       (rf.score(X_train, y_train), rf.score(X_test, y_test)),
    'KNN (k=11)':          (knn.score(X_train_sc, y_train), knn.score(X_test_sc, y_test)),
}

df_ott = pd.DataFrame(train_test_compare, index=['Train', 'Test']).T
df_ott['Diferencia'] = (df_ott['Train'] - df_ott['Test']).abs()
print(df_ott.round(4).to_string())
print("\n⚠️  Diferencia > 0.05 puede indicar overfitting")

## 6. Importancia de Variables

### 6.1 Regresión Logística – Coeficientes

In [ ]:
# Coeficientes de la Regresión Logística
coef_df = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': lr.coef_[0]
}).sort_values('Coeficiente', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_coef = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coeficiente']]
ax.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colors_coef)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 Variables – Coeficientes Regresión Logística\n(rojo=aumenta churn, verde=reduce churn)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coeficiente')
plt.tight_layout()
plt.savefig('lr_coefficients.png', dpi=120, bbox_inches='tight')
plt.show()

### 6.2 Random Forest – Importancia de Variables

In [ ]:
# Importancia de variables en Random Forest
feat_imp = pd.DataFrame({
    'Variable': X.columns,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_imp['Variable'][::-1], feat_imp['Importancia'][::-1], color='#3498db', alpha=0.85)
ax.set_title('Top 15 Variables más Importantes – Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Importancia Relativa')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print(feat_imp.round(4).to_string(index=False))

## 7. Conclusiones Estratégicas

### 7.1 Modelo Ganador

In [ ]:
print("=" * 60)
print("🏆 RESULTADOS FINALES")
print("=" * 60)
print(df_results.round(4).to_string())
print("\n🥇 Mejor modelo por F1-Score:", df_results['F1-Score'].idxmax())
print(f"   F1-Score: {df_results['F1-Score'].max():.4f}")
print(f"   Exactitud: {df_results.loc[df_results['F1-Score'].idxmax(), 'Exactitud']:.4f}")

### 7.2 Insights Estratégicos

A partir de los modelos entrenados y el análisis de importancia de variables, se identificaron los siguientes **factores clave** que influyen en la cancelación de clientes:

| Factor | Impacto | Acción Recomendada |
|--------|---------|-------------------|
| **Tipo de Contrato** | Contratos mes a mes tienen altísima tasa de churn | Ofrecer descuentos por contratos anuales o bianuales |
| **Permanencia (tenure)** | Clientes nuevos cancelan más | Programas de onboarding y fidelización en los primeros 6 meses |
| **Gasto Mensual (Monthly)** | Gastos altos correlacionan con cancelación | Revisar estructura de precios; ofrecer planes personalizados |
| **Servicio de Internet** | Fiber optic tiene mayor tasa de churn | Mejorar calidad/soporte de fibra óptica |
| **TechSupport** | Ausencia de soporte técnico aumenta churn | Incluir soporte técnico básico en todos los planes |
| **Método de Pago** | Electronic check asociado a mayor cancelación | Incentivar pagos automáticos con descuentos |

### 7.3 Recomendaciones

1. **Intervención temprana:** Identificar clientes de alto riesgo (primeros 12 meses, contrato mensual, sin soporte técnico) y activar campañas de retención proactivas.

2. **Revisión de precios en fibra óptica:** Los clientes de Fiber optic pagan más y cancelan más; revisar la relación precio-calidad-percepción.

3. **Incentivos de migración contractual:** Ofrecer beneficios atractivos para migrar de mensual a anual reduce significativamente el riesgo de churn.

4. **Modelo en producción:** El modelo **Random Forest** es recomendado para producción por su balance entre interpretabilidad y rendimiento, sin requerir normalización de datos nuevos.


In [ ]:
print("✅ Análisis completado exitosamente")
print(f"\n📁 Archivos generados:")
import os
archivos = [f for f in os.listdir('.') if f.endswith(('.png', '.csv'))]
for a in sorted(archivos):
    print(f"   - {a}")